## 第4章　向量的加减与数乘 —— 平移与缩放

> 本章目标：建立向量运算的几何直觉——加法 = 平移（平行四边形法则），数乘 = 缩放（拉长或缩短）。用 `matplotlib.quiver` 亲手画出向量运算的几何结果，并理解 AI 中最惊艳的向量运算：词嵌入的语义算术（国王 − 男人 + 女人 ≈ 女王）。
> 前置知识：第 3 章（向量、NumPy 数组、shape）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print("env ready")


### 4.1　向量加法 —— 两次"走路"的合成

你从家出发，先向东走 3 公里，再向北走 4 公里。最终位置离出发点多远？5 公里（3-4-5 直角三角形的斜边）。这个"先走 **a**，再走 **b**"的操作，就是向量加法：**a** + **b**。

📐 **定义　向量加法（Vector Addition）**：对应位置相加。若 **a** = [a₁, a₂] 和 **b** = [b₁, b₂]，则 **a** + **b** = [a₁+b₁, a₂+b₂]。几何上遵循**平行四边形法则**：以 **a** 和 **b** 为邻边画平行四边形，对角线就是 **a** + **b**。

向量加法不是"把两个箭头粘在一起"——而是**把两次平移合成为一次平移**。在 AI 中，向量加法最经典的应用是残差连接（Residual Connection）：`output = LayerNorm(x + Sublayer(x))`，那个 `x + Sublayer(x)` 就是在"原始信息"上加一个"修正量"——加法让信息可以无损地跳过一层。

💻 **代码　向量加法的代数实现与几何预览**


In [ ]:
import numpy as np

a = np.array([3.0, 0.0])   # 向东走 3 公里
b = np.array([0.0, 4.0])   # 向北走 4 公里
c = a + b                   # 合成位移

print(f"a = {a}")
print(f"b = {b}")
print(f"a + b = {c}")
print(f"合成距离 = {np.sqrt(c[0]**2 + c[1]**2):.1f} ← 恰好是 5.0（3-4-5 直角三角形）")

# 加法满足交换律和结合律——和普通数字加法一样舒服
print(f"\na + b = {a + b}")
print(f"b + a = {b + a}")
print(f"(a + b) 等于 (b + a) ? {(a + b == b + a).all()} ✓ (交换律)")

# 零向量：任何向量加零向量等于自身
zero = np.zeros(2)
print(f"\na + 0 = {a + zero} ← 等于 a 本身 (零向量是加法的'单位元')")


> **关键洞察**：向量加法如此简单（只是对应位置相加），以至于你可能会低估它在 AI 中的分量。但两个关键应用改变了这一点：(1) **残差连接**中 `x + F(x)` 让原始信号绕过变换层直达更深层——Transformer 能堆 100+ 层全靠这个加法；(2) **词嵌入语义运算**中 `国王 − 男人 + 女人` 的每一步都是向量加减——这个简单运算竟能捕捉到语义关系（第 4.4 节揭秘）。

🔗 **AI 连接**：第 29 章 Transformer 的每个 Sublayer 后面都有一个残差连接 `x = x + Sublayer(x)`——这个加法是 Transformer 能训练到上百层的根基，因为它给梯度一条"高速公路"直达底层（第 14 章反向传播将揭示其原因）。

---


### 4.2　向量数乘 —— 不转弯，只伸缩

📐 **定义　向量数乘（Scalar Multiplication）**：`c * v = [c·v₁, c·v₂, ..., c·vₙ]`。c > 1 拉长向量，0 < c < 1 缩短，c < 0 反向。数乘**不改变方向**（忽略反向情况），只缩放长度。

这比加法更简单：就是把向量的每一个分量都乘上同一个数。但在 AI 中，它扮演着两个关键角色：(1) **学习率** `lr` 就是那个"数"——`θ = θ - lr * ∇L` 中的 `lr * ∇L` 就是一个数乘，lr 控制着参数更新步长的大小；(2) **权重衰减**中 `θ *= (1 - lr * λ)` 也是数乘，让所有权重参数在每次更新前"微缩"一点点。

💻 **代码　亲手缩放：观察长度变化，方向不变**


In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
v_length = np.sqrt(np.sum(v ** 2))  # L2 范数 = 5.0

print(f"原向量 v = {v}")
print(f"v 的长度 = {v_length:.1f}  (3-4-5 直角三角形)\n")

# 不同缩放因子
for c in [2.0, 0.5, -1.0, 0.0]:
    scaled = c * v
    scaled_length = np.sqrt(np.sum(scaled ** 2))
    print(f"  c = {c:4.1f}:  {c}×v = {scaled}")
    print(f"           长度 = {scaled_length:.1f}  "
          f"(= |{c}| × {v_length:.1f} = {abs(c)*v_length:.1f})")
    # 验证方向不变：scaled 和 v 的对应分量比值相同（只要 c≠0 且分量非零）
    if c != 0:
        ratio = scaled / v
        print(f"           分量比 = {ratio} ← 所有分量缩放同样的倍数 ✓")

# 数乘的 AI 含义：学习率的效果
lr_small = 0.01
lr_large = 1.0
gradient = np.array([0.5, -0.3])
print(f"\n梯度 = {gradient}")
print(f"小学习率 0.01×梯度 = {lr_small * gradient}  ← 小步更新，训练稳定")
print(f"大学习率 1.0×梯度  = {lr_large * gradient}   ← 大步更新，可能震荡")


> **关键洞察**：数乘在几何上就是"把箭头拉长或缩短，但不转弯"。在 AI 中，这对应着一个深刻的直觉：**梯度指出了"最陡下山的方向"，学习率决定了"这一步迈多大"。** 方向由 ∇L 决定（第 12 章），步长由 lr 决定——两者通过数乘 `lr * ∇L` 结合，构成了整个深度学习训练的原子操作。

🔗 **AI 连接**：`θ -= lr * ∇L` 中的 `lr * ∇L` 贯穿第 12 章到第 30 章。第 24 章的各种优化器（Momentum、Adam、AdamW）本质上都是在"如何计算更好的更新方向"和"如何自适应调整步长"两个维度上改进这个最基础的数乘操作。

---


### 4.3　几何可视化 —— 用 quiver 亲眼看到向量运算

数字和公式只能告诉你"a + b = [4, 3]"，一张图能让你看到为什么"先向东走 3，再向北走 4"会恰好构成一个矩形，而对角线就是合成位移。

`matplotlib.quiver` 是画向量的标配工具——它接受起点坐标和方向分量，画出箭头。本节的代码会生成一张同时显示 **v**、2**v**、**v**+**u** 的图，让向量的几何直觉一目了然。

💻 **代码　quiver 绘制向量加法与数乘**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== 定义三个向量 =====
v = np.array([3.0, 1.0])   # 第一个向量
u = np.array([1.0, 2.0])   # 第二个向量
w = v + u                   # v + u（平行四边形对角线）
v2 = 2.0 * v                # 2v（v 拉长到 2 倍）

# ===== 画图 =====
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- 左图：向量加法 v + u ---
ax = axes[0]
origin = np.array([0.0, 0.0])
# quiver: X起点, Y起点, X方向, Y方向
ax.quiver(*origin, *v, angles='xy', scale_units='xy', scale=1,
          color='blue', width=0.015, label='**v**')
ax.quiver(*origin, *u, angles='xy', scale_units='xy', scale=1,
          color='green', width=0.015, label='**u**')
# 从 v 的终点再走 u：画平行四边形
ax.quiver(*v, *u, angles='xy', scale_units='xy', scale=1,
          color='green', width=0.012, alpha=0.5, linestyle='--')
ax.quiver(*u, *v, angles='xy', scale_units='xy', scale=1,
          color='blue', width=0.012, alpha=0.5, linestyle='--')
# 对角线 = v + u
ax.quiver(*origin, *w, angles='xy', scale_units='xy', scale=1,
          color='red', width=0.018, label='**v** + **u**')

ax.set_xlim(-1, 5); ax.set_ylim(-1, 4)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('向量加法：平行四边形法则')
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)
ax.legend(); ax.grid(alpha=0.3); ax.set_aspect('equal')

# --- 右图：向量数乘 2*v ---
ax = axes[1]
ax.quiver(*origin, *v, angles='xy', scale_units='xy', scale=1,
          color='blue', width=0.015, label='**v**')
ax.quiver(*origin, *v2, angles='xy', scale_units='xy', scale=1,
          color='red', width=0.015, label='2**v**')
# 标注长度
ax.annotate(f'|v|={np.sqrt(v@v):.2f}', xy=(v[0]/2, v[1]/2),
            fontsize=9, color='blue', ha='center')
ax.annotate(f'|2v|={np.sqrt(v2@v2):.2f}', xy=(v2[0]/2, v2[1]/2),
            fontsize=9, color='red', ha='center')

ax.set_xlim(-1, 7); ax.set_ylim(-1, 3)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('向量数乘：方向不变，长度翻倍')
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)
ax.legend(); ax.grid(alpha=0.3); ax.set_aspect('equal')

plt.tight_layout(); plt.show()
print("观察左图：红色对角线 = 蓝色 v + 绿色 u —— 正好构成平行四边形的对角线")
print("观察右图：红色箭头 2v 和蓝色箭头 v 指向完全相同，只是长度翻倍")


> **关键洞察**：左图中红色对角线完美诠释了"先走 v 再走 u"的几何含义——v 的终点就是"先 v 后 u"这个复合操作的起点，而 red 对角线则是从原点直达终点。**这就是残差连接**：想象 v 是原始输入 x，u 是子层输出 F(x)，红色对角线就是 `x + F(x)`——信息既经过了变换（有了 u 的方向），又没有丢失原始信号（保留了 v 的方向）。

🔗 **AI 连接**：`matplotlib.quiver` 将在第 12 章画梯度场（每个点上的箭头指向最陡上升方向），第 17 章画 PCA 的主成分方向。掌握 quiver = 掌握全书向量可视化的通用语言。

---


### 4.4　AI 应用：词嵌入的语义运算 —— 向量加减的"魔法时刻"

2013 年，Mikolov 等人的 word2vec 论文发现了一个惊人的现象：训练好的词向量中，**v("国王") − v("男人") + v("女人") ≈ v("女王")**。向量减法"去掉男性属性"，加上"女性属性"，结果落到了"女王"的附近。

这不是巧合。词嵌入训练的目标就是让语义相近的词向量也相近——结果高维空间中自动涌现出了**语义方向**：男人→女人的方向 ≈ 国王→女王的方向 ≈ 叔叔→阿姨的方向。一条向量减法，捕捉到了"性别"这个抽象概念。

📐 **定义　词嵌入语义运算**：`v(target) ≈ v(word_a) − v(context_a) + v(context_b)`。向量减法去除某属性，加法添加某属性。这证明了向量不仅是"数字列表"——高维空间中的方向可以编码语义关系。

💻 **代码　用预训练词向量验证"国王 − 男人 + 女人 ≈ 女王"**


In [ ]:
import numpy as np

# 模拟 4 维词嵌入（真实 word2vec 用 300 维，原理完全相同）
# 为让演示更直观，数据经过精心构造以展现语义关系
word_vectors = {
    "国王": np.array([0.9, 0.7, 0.5, 0.1]),
    "男人": np.array([0.6, 0.7, 0.2, 0.1]),
    "女王": np.array([0.4, 0.2, 0.6, 0.0]),
    "女人": np.array([0.1, 0.2, 0.3, 0.0]),
}

def cosine_similarity(a, b):
    """余弦相似度——衡量两个向量指向方向的接近程度（第 5 章详解）"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 语义运算：国王 − 男人 + 女人
result = word_vectors["国王"] - word_vectors["男人"] + word_vectors["女人"]

print("语义运算: 国王 − 男人 + 女人 =")
print(f"  结果向量: {result.round(3)}")
print(f"  '女王'向量: {word_vectors['女王']}")

# 计算"结果向量"与各词的余弦相似度
print(f"\n余弦相似度（越接近 1 越相似）：")
for word, vec in word_vectors.items():
    sim = cosine_similarity(result, vec)
    marker = " ← 最匹配!" if word == "女王" else ""
    print(f"  与'{word}'的相似度: {sim:.4f}{marker}")

# 验证：结果向量应该最接近"女王"
queen_sim = cosine_similarity(result, word_vectors["女王"])
king_sim = cosine_similarity(result, word_vectors["国王"])
print(f"\n结论: 结果向量更接近'女王'({queen_sim:.3f}) "
      f"而非'国王'({king_sim:.3f}) ✓")
print("这意味着: 向'国王'减去'男性'属性，加上'女性'属性 → 得到'女王'！")


> **关键洞察**：向量不仅可以表示"一个用户的特征"，还能让**方向**本身具有语义。在 300 维词嵌入空间中，"国王−男人+女人"这三步向量运算的方向组合，恰好捕捉到了"改变性别但保留王权地位"这个抽象概念。这就是向量最深刻的威力：**运算不只是数字游戏，而是对"含义"的操作。** 第 29 章 Transformer 中 Q·K 注意力得分的计算，本质上就是靠向量点积（第 5 章）来度量"query 想找什么"和"key 有什么"之间的语义相似度——而那种语义，就来自你此刻在向量运算中建立的方向直觉。

🔗 **AI 连接**：第 5 章将正式引入余弦相似度（本节预览过的 `cosine_similarity` 函数），并演示如何用向量点积做"猜你喜欢"推荐系统。第 29 章 Transformer 的 `Q @ Kᵀ` 本质上是在 512 维空间中对每对 token 做大型余弦相似度计算——你此刻看到的 4 维示例，只是把 512 换成了 4。


### 4.5　Agent 规划：状态向量与动作向量的加法 —— ReAct 的数学直觉

前面四节你学的向量加法，有一个让你意想不到的应用：**它可以直接建模 AI Agent 的"思考→行动→新状态"循环。**

在 ReAct（Reasoning + Acting）模式的 Agent 中，系统维护一个"状态向量"——编码当前的思考内容、已知信息、对话历史。当 Agent 决定执行一个动作（搜索、调用工具、反问用户），这个动作也是一个向量——描述"下一步往哪个方向走"。**状态迁移 = 当前状态向量 + 动作向量。**

用大白话说：Agent 的所有行为，不过是**在某个高维语义空间中的一次次向量平移。** 你从原点（初始状态）出发，每做一个动作就沿该动作的方向走一步，最终抵达目标状态（如找到答案、完成任务）。这和你 4.1 节"先向东走 3 公里再向北走 4 公里"是完全相同的数学——只不过维度从 2 变成了成百上千。

📐 **定义　状态迁移（State Transition）**：**s**_new = **s**_current + **a**，其中 **s** 是状态向量，**a** 是动作向量。加法充当"状态更新函数"，动作是"在状态空间中的位移向量"。

💻 **代码　用 quiver 模拟 Agent 的状态迁移轨迹**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 模拟一个 2D 语义空间中的 Agent 状态迁移
# 在实际系统中状态是几百维的，但 2D 让你看清楚"加法=平移"的几何直觉
np.random.seed(42)

# 初始状态 + 一系列动作
state = np.array([0.0, 0.0])          # Agent 初始状态（原点）
actions = [
    np.array([2.0, 1.0]),              # 动作1: "搜索数据库"
    np.array([-1.0, 2.5]),             # 动作2: "分析搜索结果"
    np.array([1.5, -1.0]),             # 动作3: "综合结论"
    np.array([0.5, 2.0]),              # 动作4: "生成回复"
]

# 轨迹：逐步累加状态
states = [state.copy()]
for a in actions:
    state = state + a                  # 状态迁移 = 向量加法！
    states.append(state.copy())
states = np.array(states)

# 可视化
fig, ax = plt.subplots(figsize=(8, 7))
colors = plt.cm.viridis(np.linspace(0, 1, len(actions)))

# 画动作箭头和状态点
for i, (s_before, a) in enumerate(zip(states[:-1], actions)):
    ax.quiver(s_before[0], s_before[1], a[0], a[1],
              angles='xy', scale_units='xy', scale=1,
              color=colors[i], width=0.03, alpha=0.8,
              label=f'Action {i+1}' if i == 0 else '')

# 画状态轨迹连线
ax.plot(states[:, 0], states[:, 1], 'ko-', markersize=8, linewidth=1.5, label='状态轨迹')
ax.plot(states[0, 0], states[0, 1], 'go', markersize=12, label='初始状态')
ax.plot(states[-1, 0], states[-1, 1], 'r*', markersize=18, label='最终状态')

# 动作标签
action_labels = ['搜索', '分析', '综合', '回复']
for i, (s_before, a) in enumerate(zip(states[:-1], actions)):
    mid = s_before + a / 2
    ax.annotate(action_labels[i], xy=(mid[0], mid[1]), fontsize=9,
                ha='center', va='bottom', color=colors[i], fontweight='bold')

ax.set_xlim(-0.5, 4.5); ax.set_ylim(-0.5, 5.5)
ax.set_xlabel('语义维度 1'); ax.set_ylabel('语义维度 2')
ax.set_title('Agent 状态迁移轨迹：每一步 = 状态向量 + 动作向量')
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_aspect('equal'); plt.show()

print("每一步: s_new = s_current + a")
print(f"起点: {states[0]} → 终点: {states[-1].round(2)}")
print(f"总位移（所有动作之和）= {actions[0]+actions[1]+actions[2]+actions[3]}")
print(f"也等于 终点 − 起点 = {states[-1] - states[0]}")
print("\n这就是 ReAct Agent 的数学本质：在语义空间中做向量加法。")


> **关键洞察**：Agent 系统中看似复杂的"规划→执行→反思"循环，在数学层面不过是**向量加法的链式调用**。`state += action` ——每一步都是一个加法，每一步都在语义空间中产生一次平移。这个直觉不仅适用于 ReAct——强化学习中的状态转移（s' = s + Δs）、扩散模型中的去噪步骤（x_{t-1} = x_t − noise_direction）、甚至 Transformer 残差连接（第 29 章）——全都可以统一为"旧状态 + 增量 = 新状态"。

🔗 **AI 连接**：第 5.5 节将把此处的状态迁移直觉扩展到 RAG 检索——用点积衡量"当前状态向量"和"候选文档向量"的匹配度。第 6.6 节从矩阵乘法视角分析连续批处理——多个用户的 Agent 状态如何并行更新。附录 D 的 Agent 速查表给出"状态迁移"的完整工程实现对照。

---


**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）用"走路"的比喻解释向量加法 a + b 的几何含义。平行四边形法则说的是什么？
2. （概念）学习率 lr 在参数更新 `θ = θ - lr * ∇L` 中执行了什么向量运算？如果 lr=0 会怎样？如果 lr 极大（如 100）会怎样？
3. （代码）画一张图，同时展示向量 a=[2,3]、b=[1,−1]、a+b、a−b、2a 五个箭头，使用 quiver，添加图例和颜色区分。
4. （代码）用 `np.random.seed(42)` 生成 5 个随机 2D 动作向量，模拟 Agent 状态迁移轨迹：从原点出发，每步执行 `state += action`，用 quiver 画出完整轨迹图（动作箭头 + 状态点连线 + 起终点标注）。
5. （代码）构造一组 6 维模拟词向量（用 `np.random.seed(42)` 保证可复现），包含"中国"、"北京"、"法国"、"巴黎"四个词。验证 `中国 − 北京 + 巴黎 ≈ 法国` 是否成立（用余弦相似度衡量，最匹配的词应该就是"法国"）。
---
> 🔗 **章末钩子**：你学会了向量的加减和数乘——但两个向量之间还有一种更深层的互动：它们指向的方向有多"一致"？这个"一致度"用一个数字来表达，就是点积。
> 预览：下一章——**向量的点积（内积）**，AI 推荐系统和 Transformer 注意力的数学核心。
